# Baseline Evaluation

Runs physics and learned baselines against the IQ L=50 dataset.
All baselines are compared on the same held-out test split.

**Metrics** (accumulated globally across the whole test set, not averaged per-batch):
- MSE in log1p(I) space
- R² in log1p(I) space
- CPU time per atom (μs), atom-count-weighted

In [ ]:
import os, subprocess, sys

NOTEBOOK_NAME = "your-kaggle-notebook"   # set to your Kaggle notebook slug
assert NOTEBOOK_NAME != "your-kaggle-notebook", "Set NOTEBOOK_NAME first."

HDF5_PATH = f"/kaggle/input/datasets/noso0s0n/iql50/I(q)L50.h5"
REPO      = f"/kaggle/working/{NOTEBOOK_NAME}"
DB_NAME   = f"{REPO}/Preprocess/iq_train_set"   # reuses the LFS-tracked iq_train_set-ENCODING.sqlite3
N_BUCKETS = 10   # number of atom-size buckets to evaluate, randomly sampled (< 57 for speed)

# install deps
%pip install -q xraydb beartype jaxtyping hdf5plugin h5py scikit-learn
!apt-get install -y -q git-lfs
!git lfs install

# clone repo
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "https://github.com/noshou/APS360.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull"], check=True)

# force-fetch real LFS content (a clone/pull before git-lfs was installed above
# would have left LFS files as small pointer stubs instead of the real blobs)
subprocess.run(["git", "-C", REPO, "lfs", "pull"], check=True)

sys.path.insert(0, REPO)

In [ ]:
import h5py, hdf5plugin
from Preprocess.encode import Encoding
from ScatterNet.utils.config import DEFAULT_BUCKETS

print("Loading encoding DB (reuses iq_train_set-ENCODING.sqlite3 from git-lfs)...")
enc = Encoding(DB_NAME, HDF5_PATH)
print(f"  {enc.count():,} molecules  |  max atoms: {enc._max}")

with h5py.File(HDF5_PATH, "r") as f:
    q_grid = f["q_grid"][()]
    energy = float(f.attrs.get("energy", 10000.0))

import torch
q_grid = torch.from_numpy(q_grid).float()
print(f"  q_grid: {len(q_grid)} points  |  energy: {energy} eV")

In [ ]:
import random
import time
from ScatterNet.batching import Batcher, Batch
from torch.utils.data import DataLoader

BUCKET_SAMPLE_SEED = 3092983   # deterministic; change to sample a different subset of buckets
LOADER_WORKERS = 4              # parallel HDF5 reads while materializing each split once

def _first(x):
    return x[0]

def _materialize(dataset, name, num_workers=LOADER_WORKERS):
    """Read every batch out of `dataset` once via a parallel DataLoader and cache
    it as a plain list. Every baseline's .fit()/evaluate() call then iterates this
    list directly instead of re-triggering per-molecule HDF5 reads from scratch
    on every single pass -- with 5 fits + 8 evaluates sharing train/test data,
    that turns ~13 full HDF5 passes into 2.
    """
    loader = DataLoader(dataset, batch_size=1, collate_fn=_first, num_workers=num_workers)
    out, t0 = [], time.time()
    for i, batch in enumerate(loader):
        out.append(batch)
        print(f"\r  materializing {name}: {i+1}/{len(dataset)}  ({time.time()-t0:.0f}s elapsed)",
              end="", flush=True)
    print()
    return out

eval_buckets = sorted(
    random.Random(BUCKET_SAMPLE_SEED).sample(DEFAULT_BUCKETS, min(N_BUCKETS, len(DEFAULT_BUCKETS)))
)

batcher = Batcher(
    hdf5_db        = HDF5_PATH,
    enc            = enc,
    batches        = eval_buckets,
    seed           = 42,
    atom_size_ceil = 6046,
)
_, _, test_set = batcher.get_sets()

test_loader = _materialize(test_set, "test set")
print(f"Test batches: {len(test_loader)}")

In [ ]:
def evaluate(baseline, loader, name):
    """Evaluate a baseline on loader. Returns (mse, r2, us_per_atom).

    Accumulates sum-of-squares globally across the whole test set (not
    per-batch) before computing MSE/R² once at the end. Buckets vary hugely
    in molecule count and target variance (a handful of huge molecules vs.
    thousands of tiny ones), so averaging a per-batch R² across buckets lets
    one low-variance bucket (small ss_tot) swing the whole score wildly --
    this instead matches the standard, statistically stable R² definition.
    """
    sum_sq_err  = 0.0
    sum_y       = 0.0
    sum_y2      = 0.0
    n_points    = 0
    tpa_weighted = 0.0
    total_atoms  = 0

    for batch in loader:
        pred, tpa = baseline.timed_call(batch)
        target = torch.log1p(batch.iqval)
        log_pred = torch.log1p(pred.clamp(min=0))
        sq_err = (log_pred - target) ** 2

        sum_sq_err += sq_err.sum().item()
        sum_y      += target.sum().item()
        sum_y2     += (target ** 2).sum().item()
        n_points   += target.numel()

        n_atoms = int(batch.padding_mask().sum().item())
        tpa_weighted += tpa * n_atoms
        total_atoms  += n_atoms

    mse = sum_sq_err / n_points if n_points > 0 else float('nan')
    ss_tot = sum_y2 - (sum_y ** 2) / n_points if n_points > 0 else 0.0
    r2 = 1 - sum_sq_err / ss_tot if ss_tot > 0 else float('nan')
    tpa_us = (tpa_weighted / total_atoms * 1e6) if total_atoms > 0 else float('nan')

    print(f"{name:<30s}  MSE={mse:.4f}  R²={r2:.4f}  {tpa_us:.2f} μs/atom")
    return mse, r2, tpa_us

In [ ]:
sys.path.insert(0, f"{REPO}/Baselines/physics-benchmarks")
sys.path.insert(0, f"{REPO}/Baselines/learned-benchmarks")

from rg import RgBaseline, GuinierPorodBaseline
from atom_count import AtomCountBaseline
from composition_regression import CompositionRegressionBaseline
from pair_peak import BinnedDebyeBaseline

print("=== Physics Baselines ===")
results = {}

# fit on train set where needed -- materialized once, reused by every .fit() below
# and by the learned baselines further down, instead of re-reading HDF5 per call
train_set, _, _ = batcher.get_sets()
train_loader = _materialize(train_set, "train set")

for name, baseline in [
    ("Guinier (Rg)",          RgBaseline(q_grid, energy)),
    ("Guinier-Porod",         GuinierPorodBaseline(q_grid, energy)),
    ("Atom Count",            AtomCountBaseline().fit(train_loader)),
    ("Composition Regression",CompositionRegressionBaseline().fit(train_loader)),
    ("Binned Debye",          BinnedDebyeBaseline(q_grid, energy)),
]:
    results[name] = evaluate(baseline, test_loader, name)

In [ ]:
import torch
import torch.nn as nn
from Preprocess import VOCAB
from Baselines.baseline import Baseline

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


class TorchMlp(Baseline):
    """GPU-accelerated MLP (replaces sklearn Mlp; targets Kaggle P100/T4)."""

    def __init__(self, hidden=(64, 64), lr=1e-3, epochs=200, mini_batch=256, grad_clip=1.0):
        self.hidden     = hidden
        self.lr         = lr
        self.epochs     = epochs
        self.mini_batch = mini_batch
        self.grad_clip  = grad_clip
        self._net    = None
        self._x_mean = None
        self._x_std  = None

    def _features(self, batch):
        N, M = batch.vocab.shape
        V = len(VOCAB) + 1
        counts = torch.zeros(N, V).scatter_add_(1, batch.vocab.long(), torch.ones(N, M))
        counts[:, 0] = 0.0
        n_atoms = counts.sum(dim=1, keepdim=True).clamp(min=1)
        mask = batch.padding_mask().float()
        r2   = (batch.coord ** 2).sum(dim=-1)
        rg   = ((r2 * mask).sum(dim=1, keepdim=True) / n_atoms).clamp(min=0).sqrt()
        return torch.cat([counts / n_atoms, n_atoms, rg], dim=1).cpu()

    def fit(self, loader):
        X_parts, Y_parts = [], []
        for batch in loader:
            X_parts.append(self._features(batch))
            Y_parts.append(torch.log1p(batch.iqval).cpu())
        X = torch.cat(X_parts)
        Y = torch.cat(Y_parts)
        self._x_mean = X.mean(0)
        self._x_std  = X.std(0).clamp(min=1e-8)
        X = (X - self._x_mean) / self._x_std
        in_dim, out_dim = X.shape[1], Y.shape[1]
        layers, prev = [], in_dim
        for h in self.hidden:
            layers += [nn.Linear(prev, h), nn.ReLU()]
            prev = h
        layers.append(nn.Linear(prev, out_dim))
        self._net = nn.Sequential(*layers).to(DEVICE)
        X, Y = X.to(DEVICE), Y.to(DEVICE)
        opt  = torch.optim.Adam(self._net.parameters(), lr=self.lr)
        n    = len(X)
        self._net.train()
        for _ in range(self.epochs):
            perm = torch.randperm(n, device=DEVICE)
            for i in range(0, n, self.mini_batch):
                b    = perm[i:i + self.mini_batch]
                loss = nn.functional.mse_loss(self._net(X[b]), Y[b])
                opt.zero_grad()
                loss.backward()
                # heavy-tailed features (atom counts / Rg span ~1 to 6046 atoms) can
                # produce occasional large gradients that blow Adam up to inf/nan
                # over 200 epochs; clip so one bad minibatch can't diverge the run.
                nn.utils.clip_grad_norm_(self._net.parameters(), self.grad_clip)
                opt.step()
        self._net.eval()
        return self

    def __call__(self, batch):
        X = (self._features(batch) - self._x_mean) / self._x_std
        with torch.no_grad():
            log_pred = self._net(X.to(DEVICE))
        # safety net: clamp before expm1 so a NaN/exploded weight (despite grad
        # clipping) surfaces as a large finite MSE instead of inf, which would
        # otherwise break evaluate()'s accumulated sums and the summary plot.
        log_pred = torch.nan_to_num(log_pred, nan=0.0, posinf=30.0, neginf=0.0).clamp(max=30.0)
        return torch.expm1(log_pred).cpu()

    def timed_call(self, batch):
        X = (self._features(batch) - self._x_mean) / self._x_std
        X = X.to(DEVICE)
        if DEVICE.type == "cuda":
            start = torch.cuda.Event(enable_timing=True)
            end   = torch.cuda.Event(enable_timing=True)
            start.record()
            with torch.no_grad():
                log_pred = self._net(X)
            end.record()
            torch.cuda.synchronize()
            elapsed = start.elapsed_time(end) / 1000.0
        else:
            import time
            t0 = time.process_time()
            with torch.no_grad():
                log_pred = self._net(X)
            elapsed = time.process_time() - t0
        log_pred = torch.nan_to_num(log_pred, nan=0.0, posinf=30.0, neginf=0.0).clamp(max=30.0)
        pred    = torch.expm1(log_pred).cpu()
        n_atoms = int(batch.padding_mask().sum().item())
        return pred, elapsed / max(n_atoms, 1)


In [ ]:
from linsvm import Linsvm
from nearest_neighbour import NNBaseline

print("=== Learned Baselines ===")

for name, baseline in [
    ("MLP",                TorchMlp().fit(train_loader)),
    ("Linear SVM",         Linsvm().fit(train_loader)),
    ("Nearest Neighbour",  NNBaseline(q_grid, energy).fit(train_loader)),
]:
    results[name] = evaluate(baseline, test_loader, name)

In [ ]:
print("\n=== Summary ===")
print(f"{'Baseline':<30s}  {'MSE':>8s}  {'R²':>8s}  {'μs/atom':>10s}")
print("-" * 62)
for name, (mse, r2, tpa) in sorted(results.items(), key=lambda x: x[1][0]):
    print(f"{name:<30s}  {mse:>8.4f}  {r2:>8.4f}  {tpa:>10.2f}")

In [ ]:
import math
import matplotlib.pyplot as plt

# fixed categorical order (identity per baseline), never reassigned by rank
PALETTE = [
    "#2a78d6", "#1baf7a", "#eda100", "#008300",
    "#4a3aa7", "#e34948", "#e87ba4", "#eb6834"
]
TEXT_PRIMARY, TEXT_MUTED, GRID = "#0b0b0b", "#898781", "#e1e0d9"

ordered = sorted(results.items(), key=lambda x: x[1][0])  # ranked by MSE, best first
names   = [n for n, _ in ordered]
colors  = [PALETTE[i % len(PALETTE)] for i in range(len(ordered))]
mse_v   = [v[0] for _, v in ordered]
r2_v    = [v[1] for _, v in ordered]
tpa_v   = [v[2] for _, v in ordered]

fig, axes = plt.subplots(1, 3, figsize=(15, 0.45 * len(ordered) + 1.5))
specs = [
    (axes[0], mse_v, "MSE (log1p I)", "{:.4f}"),
    (axes[1], r2_v,  "R²",            "{:.3f}"),
    (axes[2], tpa_v, "μs / atom",     "{:.1f}"),
]

y = range(len(ordered))
for ax, vals, title, fmt in specs:
    # NaN/Inf are possible (e.g. R² is NaN when a molecule's ss_tot == 0, or an
    # under-trained baseline predicts values that blow up log1p) -- draw those
    # bars as zero-width with an "n/a" label instead of feeding non-finite
    # numbers into matplotlib, which raises on set_xlim.
    finite_vals = [v for v in vals if math.isfinite(v)]
    xmax = max(finite_vals) if finite_vals else 1
    plot_vals = [v if math.isfinite(v) else 0 for v in vals]

    bars = ax.barh(y, plot_vals, color=colors, height=0.6, zorder=3)
    ax.set_yticks(list(y))
    ax.set_yticklabels(names if ax is axes[0] else [], color=TEXT_PRIMARY)
    ax.invert_yaxis()  # best (lowest MSE) at top, matches sort order
    ax.set_title(title, color=TEXT_PRIMARY, fontsize=11)
    ax.tick_params(colors=TEXT_MUTED, length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.grid(axis="x", color=GRID, linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    for bar, v in zip(bars, vals):
        label = "n/a" if not math.isfinite(v) else fmt.format(v)
        ax.text(bar.get_width() + 0.02 * xmax, bar.get_y() + bar.get_height() / 2,
                label, va="center", ha="left", fontsize=8.5, color=TEXT_PRIMARY)
    ax.set_xlim(0, xmax * 1.18 if xmax > 0 else 1)

fig.suptitle("Baseline comparison (sorted by MSE, best first)", color=TEXT_PRIMARY, fontsize=12)
fig.tight_layout(rect=(0, 0, 1, 0.95))
plt.show()